In [1]:
df = spark.read.parquet(
    "s3://airline-dataset-2020-2025/Gold/ML_DATASET/"
)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1785306838705_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
drop_cols = [
    "FlightKey",
    "FlightDate",
    "ReliabilityFeatureScope"
]

df = df.drop(*drop_cols)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
from pyspark.sql.functions import mean

cols = [
    "OriginAirportReliabilityScore",
    "DestAirportReliabilityScore",
    "RouteReliabilityScore"
]

fill_values = {}

for c in cols:
    fill_values[c] = df.select(mean(c)).first()[0]

df = df.na.fill(fill_values)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
categorical_cols = [
    "MarketingAirlineKey",
    "OperatingAirlineKey",
    "DeparturePeriod",
    "ArrivalPeriod",
    "SeasonIndicator",
    "DistanceCategory"
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
numeric_cols = [
    "DepartureHour",
    "ArrivalHour",
    "PeakHourIndicator",
    "WeekendIndicator",
    "Distance",
    "ScheduledElapsedTimeMinutes",
    "CodeshareFlag",
    "IntraStateRouteFlag",
    "AirlineReliabilityScore",
    "OriginAirportReliabilityScore",
    "DestAirportReliabilityScore",
    "RouteReliabilityScore"
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=numeric_cols + [c + "_idx" for c in categorical_cols],
    outputCol="features",
    handleInvalid="skip"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=indexers + [assembler]
)

pipeline_model = pipeline.fit(df)

processed_df = pipeline_model.transform(df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
sample_df = processed_df.sample(
    withReplacement=False,
    fraction=0.1,
    seed=42
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
train_df = sample_df.filter("DatasetSplit = 'Train'")
valid_df = sample_df.filter("DatasetSplit = 'Validation'")
test_df  = sample_df.filter("DatasetSplit = 'Test'")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(
    labelCol="ArrDel15",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    maxBins=64,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model = rf.fit(train_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
feature_names = numeric_cols + [c + "_idx" for c in categorical_cols]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
importance = rf_model.featureImportances.toArray()

feature_importance = list(zip(feature_names, importance))

feature_importance = sorted(
    feature_importance,
    key=lambda x: x[1],
    reverse=True
)

for feature, score in feature_importance:
    print(f"{feature:<40} {score:.6f}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RouteReliabilityScore                    0.179712
DepartureHour                            0.175234
DeparturePeriod_idx                      0.116260
SeasonIndicator_idx                      0.115219
ArrivalHour                              0.110418
AirlineReliabilityScore                  0.097349
ArrivalPeriod_idx                        0.059244
OriginAirportReliabilityScore            0.031835
OperatingAirlineKey_idx                  0.028053
MarketingAirlineKey_idx                  0.027047
DestAirportReliabilityScore              0.018373
CodeshareFlag                            0.017803
ScheduledElapsedTimeMinutes              0.007281
Distance                                 0.007139
PeakHourIndicator                        0.006057
DistanceCategory_idx                     0.001157
WeekendIndicator                         0.001060
IntraStateRouteFlag                      0.000758

In [14]:
selected_features = [
    feature
    for feature, score in feature_importance
    if score > 0.01
]

print(selected_features)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['RouteReliabilityScore', 'DepartureHour', 'DeparturePeriod_idx', 'SeasonIndicator_idx', 'ArrivalHour', 'AirlineReliabilityScore', 'ArrivalPeriod_idx', 'OriginAirportReliabilityScore', 'OperatingAirlineKey_idx', 'MarketingAirlineKey_idx', 'DestAirportReliabilityScore', 'CodeshareFlag']

In [51]:
selected_features = [
    "RouteReliabilityScore",
    "DepartureHour",
    "DeparturePeriod_idx",
    "SeasonIndicator_idx",
    "ArrivalHour",
    "AirlineReliabilityScore",
    "ArrivalPeriod_idx",
    "OriginAirportReliabilityScore",
    "OperatingAirlineKey_idx",
    "MarketingAirlineKey_idx",
    "DestAirportReliabilityScore",
    "CodeshareFlag",
    "Distance",
    "PeakHourIndicator",
    "WeekendIndicator",
    "ScheduledElapsedTimeMinutes",
    "DayOfWeek",
    "Month",
]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [52]:
from pyspark.ml.feature import VectorAssembler

final_assembler = VectorAssembler(
    inputCols=selected_features,
    outputCol="selected_features"
)

final_df = final_assembler.transform(processed_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [53]:
from pyspark import StorageLevel

final_df.persist(StorageLevel.MEMORY_AND_DISK)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[RouteKey: string, DestAirportKey: string, OriginAirportKey: string, MarketingAirlineKey: string, Quarter: int, DayofMonth: int, DayOfWeek: int, DepartureHour: int, ArrivalHour: int, DeparturePeriod: string, ArrivalPeriod: string, PeakHourIndicator: int, WeekendIndicator: int, SeasonIndicator: string, OperatingAirlineKey: string, Distance: int, ScheduledElapsedTimeMinutes: int, DistanceCategory: string, CodeshareFlag: int, IntraStateRouteFlag: int, ArrDel15: int, AirlineReliabilityScore: double, OriginAirportReliabilityScore: double, DestAirportReliabilityScore: double, RouteReliabilityScore: double, DatasetSplit: string, Year: int, Month: int, MarketingAirlineKey_idx: double, OperatingAirlineKey_idx: double, DeparturePeriod_idx: double, ArrivalPeriod_idx: double, SeasonIndicator_idx: double, DistanceCategory_idx: double, features: vector, selected_features: vector]

In [54]:
train_df = final_df.filter(processed_df.DatasetSplit == "Train")

valid_df = final_df.filter(processed_df.DatasetSplit == "Validation")

test_df = final_df.filter(processed_df.DatasetSplit == "Test")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
train_df.cache()
valid_df.cache()
test_df.cache()

In [55]:
train_df = train_df.drop("DatasetSplit")
valid_df = valid_df.drop("DatasetSplit")
test_df = test_df.drop("DatasetSplit")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [59]:
train_df = train_df.repartition(256)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [60]:
rf = RandomForestClassifier(
    labelCol="ArrDel15",
    featuresCol="selected_features",
    numTrees=150,
    maxDepth=10,
    maxBins=64,
    seed=42
)

rf_model = rf.fit(train_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [61]:
predictions = rf_model.transform(test_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [62]:
predictions.select(
    "ArrDel15",
    "prediction",
    "probability"
).show(10, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+----------+----------------------------------------+
|ArrDel15|prediction|probability                             |
+--------+----------+----------------------------------------+
|0       |0.0       |[0.8828495014754221,0.11715049852457796]|
|0       |0.0       |[0.7770528648975027,0.22294713510249728]|
|0       |0.0       |[0.715464335906863,0.28453566409313696] |
|0       |0.0       |[0.7190634540622052,0.2809365459377949] |
|0       |0.0       |[0.8828495014754221,0.11715049852457796]|
|0       |0.0       |[0.7094112495800675,0.29058875041993254]|
|0       |0.0       |[0.7190634540622052,0.2809365459377949] |
|0       |0.0       |[0.8828495014754221,0.11715049852457796]|
|0       |0.0       |[0.7094112495800675,0.29058875041993254]|
|0       |0.0       |[0.7190634540622052,0.2809365459377949] |
+--------+----------+----------------------------------------+
only showing top 10 rows

In [63]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="accuracy"
)

print("Accuracy:", accuracy.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Accuracy: 0.7782182153459791

In [64]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc = BinaryClassificationEvaluator(
    labelCol="ArrDel15",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

print("ROC AUC:", auc.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

ROC AUC: 0.6429255066435244

In [65]:
f1 = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="f1"
)

print("F1 Score:", f1.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

F1 Score: 0.6825541705806899

In [68]:
precision = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

print("Precision:", precision.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Precision: 0.7202281672021356

In [66]:
recall = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="weightedRecall"
)

print("Recall:", recall.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Recall: 0.7782182153459792

In [67]:
predictions.groupBy("ArrDel15", "prediction").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+----------+-------+
|ArrDel15|prediction|  count|
+--------+----------+-------+
|       1|       1.0|   5813|
|       0|       0.0|5906696|
|       0|       1.0|   5465|
|       1|       0.0|1679521|
+--------+----------+-------+

In [73]:
train_df.groupBy("ArrDel15").count().show()

valid_df.groupBy("ArrDel15").count().show()

test_df.groupBy("ArrDel15").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+--------+
|ArrDel15|   count|
+--------+--------+
|       1| 4427858|
|       0|20444792|
+--------+--------+

+--------+-------+
|ArrDel15|  count|
+--------+-------+
|       1|1531080|
|       0|5894149|
+--------+-------+

+--------+-------+
|ArrDel15|  count|
+--------+-------+
|       1|1685334|
|       0|5912161|
+--------+-------+

In [74]:
predictions.select(
    "probability",
    "prediction",
    "ArrDel15"
).show(20, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------------------------+----------+--------+
|probability                             |prediction|ArrDel15|
+----------------------------------------+----------+--------+
|[0.8828495014754221,0.11715049852457796]|0.0       |0       |
|[0.7770528648975027,0.22294713510249728]|0.0       |0       |
|[0.715464335906863,0.28453566409313696] |0.0       |0       |
|[0.7190634540622052,0.2809365459377949] |0.0       |0       |
|[0.8828495014754221,0.11715049852457796]|0.0       |0       |
|[0.7094112495800675,0.29058875041993254]|0.0       |0       |
|[0.7190634540622052,0.2809365459377949] |0.0       |0       |
|[0.8828495014754221,0.11715049852457796]|0.0       |0       |
|[0.7094112495800675,0.29058875041993254]|0.0       |0       |
|[0.7190634540622052,0.2809365459377949] |0.0       |0       |
|[0.8827700932182956,0.11722990678170452]|0.0       |0       |
|[0.7094112495800675,0.29058875041993254]|0.0       |1       |
|[0.8828495014754221,0.11715049852457796]|0.0       |0 

In [75]:
train_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- RouteKey: string (nullable = true)
 |-- DestAirportKey: string (nullable = true)
 |-- OriginAirportKey: string (nullable = true)
 |-- MarketingAirlineKey: string (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepartureHour: integer (nullable = true)
 |-- ArrivalHour: integer (nullable = true)
 |-- DeparturePeriod: string (nullable = true)
 |-- ArrivalPeriod: string (nullable = true)
 |-- PeakHourIndicator: integer (nullable = true)
 |-- WeekendIndicator: integer (nullable = true)
 |-- SeasonIndicator: string (nullable = true)
 |-- OperatingAirlineKey: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- ScheduledElapsedTimeMinutes: integer (nullable = true)
 |-- DistanceCategory: string (nullable = true)
 |-- CodeshareFlag: integer (nullable = true)
 |-- IntraStateRouteFlag: integer (nullable = true)
 |-- ArrDel15: integer (nullable = true)
 |-- AirlineReliab

In [76]:
train_df.select("selected_features").show(5, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------------------------------------------------------------------------------------+
|selected_features                                                                        |
+-----------------------------------------------------------------------------------------+
|[90.79,7.0,0.0,3.0,10.0,90.74,2.0,88.52,1.0,1.0,85.78,0.0,954.0,1.0,1.0,180.0,6.0,12.0]  |
|[88.84,20.0,2.0,1.0,21.0,87.47,1.0,88.76,15.0,0.0,84.77,1.0,413.0,0.0,1.0,103.0,6.0,10.0]|
|[89.24,20.0,2.0,1.0,21.0,87.47,1.0,88.76,10.0,0.0,86.74,1.0,490.0,0.0,1.0,100.0,7.0,9.0] |
|[85.9,9.0,0.0,0.0,10.0,87.57,2.0,88.35,3.0,3.0,82.26,1.0,627.0,1.0,0.0,128.0,1.0,6.0]    |
|[86.45,10.0,0.0,0.0,12.0,86.12,0.0,86.87,0.0,2.0,80.34,0.0,721.0,0.0,0.0,125.0,5.0,6.0]  |
+-----------------------------------------------------------------------------------------+
only showing top 5 rows

In [77]:
print(train_df.select("selected_features").first()[0].size)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

18

In [82]:
predictions.select(
    col("probability")[1].alias("p_delay"),
    "ArrDel15",
    "prediction"
).orderBy(col("p_delay").desc()).show(30, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
[INVALID_EXTRACT_BASE_FIELD_TYPE] Can't extract a value from "probability". Need a complex type [STRUCT, ARRAY, MAP] but got "STRUCT<type: TINYINT, size: INT, indices: ARRAY<INT>, values: ARRAY<DOUBLE>>".
Traceback (most recent call last):
  File "/mnt/yarn/usercache/livy/appcache/application_1785306838705_0001/container_1785306838705_0001_01_000001/pyspark.zip/pyspark/sql/dataframe.py", line 3037, in select
    jdf = self._jdf.select(self._jcols(*cols))
  File "/mnt/yarn/usercache/livy/appcache/application_1785306838705_0001/container_1785306838705_0001_01_000001/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1323, in __call__
    answer, self.gateway_client, self.target_id, self.name)
  File "/mnt/yarn/usercache/livy/appcache/application_1785306838705_0001/container_1785306838705_0001_01_000001/pyspark.zip/pyspark/errors/exceptions/captured.py", line 175, in deco
    raise converted from None
pyspark.errors.exceptions.captured.AnalysisException: [INVALID_

In [84]:
model_path = "s3://raj-617.01/models"

rf_model.write().overwrite().save(model_path)

print("Model saved successfully to S3!")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model saved successfully to S3!

In [85]:
from pyspark.ml.classification import RandomForestClassificationModel

model_path = "s3://raj-617.01/models"

rf_model = RandomForestClassificationModel.load(model_path)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [86]:
predictions = rf_model.transform(test_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [87]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="ArrDel15",
    predictionCol="prediction",
    metricName="accuracy"
)

print("Accuracy:", accuracy.evaluate(predictions))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Accuracy: 0.7782182153459791